In [26]:

import json
from pathlib import Path
import joblib
import numpy as np
import pandas as pd

# Location where we saved our previous calculations.
checkpoint_dir = Path(
    r"C:\Users\karra\Desktop\notebook12_checkpoints"
)

# Read the list of saved variables.
with open(checkpoint_dir / "manifest.json", "r") as file:
    manifest = json.load(file)

# Restore all previously saved variables automatically.
restored_variables = {}

for name, filename in manifest["variables"].items():
    restored_variables[name] = joblib.load(
        checkpoint_dir / filename
    )

# Make the restored variables available in Notebook 12.
globals().update(restored_variables)

print("CHECKPOINT RESTORED!")
print("Variables restored:", len(restored_variables))

# Verify our important datasets and final investigation.
for name in [
    "sales",
    "calendar",
    "prices",
    "activation_144",
    "activation_audit_final"
]:
    print(
        name,
        ":",
        "AVAILABLE" if name in globals() else "MISSING"
    )


CHECKPOINT RESTORED!
Variables restored: 427
sales : AVAILABLE
calendar : AVAILABLE
prices : AVAILABLE
activation_144 : AVAILABLE
activation_audit_final : AVAILABLE


In [9]:

# Check which important variables are available in memory.
for variable in [
    "sales",
    "calendar",
    "prices",
    "price_audit",
    "events_corrected",
    "low_price_episodes"
]:
    print(
        variable,
        ":", "AVAILABLE" if variable in globals() else "MISSING"
    )


sales : AVAILABLE
calendar : AVAILABLE
prices : AVAILABLE
price_audit : MISSING
events_corrected : MISSING
low_price_episodes : MISSING


In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

# Notebook is inside /notebooks, so go one level up to the project root.
PROJECT_ROOT = Path("..")

RAW_DIR = PROJECT_ROOT / "data" / "m5" / "raw"
PROCESSED_DIR = PROJECT_ROOT / "data" / "m5" / "processed"

print("Raw data folder:", RAW_DIR.resolve())
print("Processed data folder:", PROCESSED_DIR.resolve())

# Show exactly what files we downloaded before loading anything.
print("\nFiles in raw folder:")
for file in sorted(RAW_DIR.iterdir()):
    print(f"  {file.name}")

In [ ]:
# Check the size of every raw M5 file before loading them into memory.
for file in sorted(RAW_DIR.glob("*.csv")):
    size_mb = file.stat().st_size / (1024 ** 2)
    print(f"{file.name:<30} {size_mb:>8.2f} MB")

In [ ]:
# Load the three core M5 datasets.
# We are deliberately NOT loading the validation or submission files yet.
sales = pd.read_csv(RAW_DIR / "sales_train_evaluation.csv")
calendar = pd.read_csv(RAW_DIR / "calendar.csv")
prices = pd.read_csv(RAW_DIR / "sell_prices.csv")

print("Sales shape:   ", sales.shape)
print("Calendar shape:", calendar.shape)
print("Prices shape:  ", prices.shape)

In [ ]:
# ============================================================
# INSPECT THE THREE CORE M5 TABLES
# ============================================================

print("SALES COLUMNS:")
print(sales.columns[:20].tolist())

print("\nSALES SAMPLE:")
display(sales.iloc[:3, :15])

print("\nCALENDAR COLUMNS:")
print(calendar.columns.tolist())

print("\nCALENDAR SAMPLE:")
display(calendar.head(3))

print("\nPRICE COLUMNS:")
print(prices.columns.tolist())

print("\nPRICE SAMPLE:")
display(prices.head(3))

In [ ]:
# ============================================================
# BASIC M5 DATASET AUDIT
# ============================================================

day_cols = [col for col in sales.columns if col.startswith("d_")]

print("Number of item-store series:", len(sales))
print("Number of daily sales columns:", len(day_cols))

print("\nUnique items:", sales["item_id"].nunique())
print("Unique stores:", sales["store_id"].nunique())
print("Unique states:", sales["state_id"].nunique())
print("Unique departments:", sales["dept_id"].nunique())
print("Unique categories:", sales["cat_id"].nunique())

print("\nFirst sales day:", day_cols[0])
print("Last sales day:", day_cols[-1])

print("\nDuplicate series IDs:", sales["id"].duplicated().sum())

print("\nMissing values:")
print("Sales:", sales.isna().sum().sum())
print("Calendar:", calendar.isna().sum().sum())
print("Prices:", prices.isna().sum().sum())

print("\nCalendar columns:")
print(calendar.columns.tolist())

In [ ]:
# ============================================================
# SALES ACTIVITY / ZERO-DEMAND AUDIT
# ============================================================

sales_values = sales[day_cols]

total_values = sales_values.size
zero_values = (sales_values == 0).sum().sum()
positive_values = (sales_values > 0).sum().sum()
negative_values = (sales_values < 0).sum().sum()

print("Total daily sales observations:", f"{total_values:,}")
print("Zero observations:", f"{zero_values:,}",
      f"({zero_values / total_values:.2%})")
print("Positive observations:", f"{positive_values:,}",
      f"({positive_values / total_values:.2%})")
print("Negative observations:", f"{negative_values:,}")

# For each item-store series, find the first and last day with positive sales.
has_positive_sales = sales_values.gt(0)

first_positive_position = has_positive_sales.values.argmax(axis=1)

# Series that never record a positive sale need to be identified separately.
never_positive = ~has_positive_sales.any(axis=1)

first_positive_day = pd.Series(
    first_positive_position + 1,
    index=sales.index
)

first_positive_day[never_positive] = np.nan

print("\nSeries with no positive sales:", never_positive.sum())

print("\nFirst positive-sale day distribution:")
print(first_positive_day.describe())

print("\nSeries whose first positive sale occurs after:")
for day in [30, 90, 180, 365, 730, 1000]:
    count = (first_positive_day > day).sum()
    print(f"d_{day:<4}: {count:>6,} series ({count / len(sales):.2%})")

In [ ]:
# ============================================================
# ITEM AVAILABILITY AUDIT
# Compare first recorded price with first positive sale
# ============================================================

# First week in which each store-item combination has a recorded price.
first_price_week = (
    prices.groupby(["store_id", "item_id"])["wm_yr_wk"]
    .min()
    .reset_index(name="first_price_week")
)

# Calendar tells us which daily M5 day belongs to each Walmart week.
day_to_week = calendar[["d", "wm_yr_wk"]].copy()

# Extract numeric day number: d_1 -> 1, d_1941 -> 1941.
day_to_week["day_num"] = (
    day_to_week["d"]
    .str.replace("d_", "", regex=False)
    .astype(int)
)

# Find the first calendar day belonging to each Walmart week.
week_first_day = (
    day_to_week.groupby("wm_yr_wk")["day_num"]
    .min()
    .reset_index(name="first_price_day")
)

first_price = first_price_week.merge(
    week_first_day,
    left_on="first_price_week",
    right_on="wm_yr_wk",
    how="left"
)

# Build one row per sales series.
availability_audit = sales[
    ["id", "store_id", "item_id"]
].copy()

availability_audit["first_positive_day"] = first_positive_day

availability_audit = availability_audit.merge(
    first_price[
        ["store_id", "item_id", "first_price_week", "first_price_day"]
    ],
    on=["store_id", "item_id"],
    how="left"
)

availability_audit["days_price_before_first_sale"] = (
    availability_audit["first_positive_day"]
    - availability_audit["first_price_day"]
)

print("Series:", len(availability_audit))

print(
    "Series without any recorded price:",
    availability_audit["first_price_week"].isna().sum()
)

print("\nFirst price day:")
print(availability_audit["first_price_day"].describe())

print("\nDays from first recorded price to first positive sale:")
print(
    availability_audit["days_price_before_first_sale"].describe()
)

print("\nSample:")
display(availability_audit.head(10))

In [ ]:
# ============================================================
# INVESTIGATE PRICE-TO-FIRST-SALE OUTLIERS
# ============================================================

gap = availability_audit["days_price_before_first_sale"]

print("Gap distribution:")
for days in [0, 7, 14, 30, 90, 180, 365]:
    count = (gap > days).sum()
    print(
        f"More than {days:>3} days: "
        f"{count:>6,} series ({count / len(availability_audit):.2%})"
    )

print("\nLargest gaps:")
display(
    availability_audit
    .sort_values("days_price_before_first_sale", ascending=False)
    .head(20)
)

In [ ]:
# ============================================================
# ZERO-DEMAND RATE DURING ACTIVE ITEM-STORE PERIODS
# ============================================================

active_zero_count = 0
active_positive_count = 0
active_observation_count = 0
pre_active_observation_count = 0

sales_array = sales[day_cols].to_numpy()

for i, first_day in enumerate(availability_audit["first_price_day"]):
    # Convert M5 day number (d_1 = 1) to Python's zero-based index.
    start_idx = int(first_day) - 1

    series_active_sales = sales_array[i, start_idx:]

    active_observation_count += len(series_active_sales)
    pre_active_observation_count += start_idx

    active_zero_count += np.sum(series_active_sales == 0)
    active_positive_count += np.sum(series_active_sales > 0)

print("Total observations:", f"{sales_array.size:,}")

print(
    "Pre-availability observations:",
    f"{pre_active_observation_count:,}",
    f"({pre_active_observation_count / sales_array.size:.2%})"
)

print(
    "Active observations:",
    f"{active_observation_count:,}",
    f"({active_observation_count / sales_array.size:.2%})"
)

print(
    "\nActive zero-demand observations:",
    f"{active_zero_count:,}",
    f"({active_zero_count / active_observation_count:.2%})"
)

print(
    "Active positive-demand observations:",
    f"{active_positive_count:,}",
    f"({active_positive_count / active_observation_count:.2%})"
)

In [ ]:
# ============================================================
# PRICE AVAILABILITY AFTER PRODUCT ACTIVATION
# ============================================================

# Number of distinct weeks in which each store-item has a price.
price_summary = (
    prices.groupby(["store_id", "item_id"])
    .agg(
        first_price_week=("wm_yr_wk", "min"),
        last_price_week=("wm_yr_wk", "max"),
        weeks_with_price=("wm_yr_wk", "nunique")
    )
    .reset_index()
)

# Give every Walmart week an ordered position:
# first week = 0, next week = 1, etc.
all_weeks = (
    calendar[["wm_yr_wk"]]
    .drop_duplicates()
    .sort_values("wm_yr_wk")
    .reset_index(drop=True)
)

all_weeks["week_position"] = np.arange(len(all_weeks))

week_position_map = dict(
    zip(all_weeks["wm_yr_wk"], all_weeks["week_position"])
)

# Convert first/last week into positions so we can calculate
# how many weeks should exist between them.
price_summary["first_week_position"] = (
    price_summary["first_price_week"].map(week_position_map)
)

price_summary["last_week_position"] = (
    price_summary["last_price_week"].map(week_position_map)
)

price_summary["expected_weeks"] = (
    price_summary["last_week_position"]
    - price_summary["first_week_position"]
    + 1
)

# Missing weeks inside the active price period.
price_summary["missing_price_weeks"] = (
    price_summary["expected_weeks"]
    - price_summary["weeks_with_price"]
)

print("Store-item series:", len(price_summary))

print(
    "Series with at least one internal missing-price week:",
    (price_summary["missing_price_weeks"] > 0).sum()
)

print("\nMissing-price-week distribution:")
print(price_summary["missing_price_weeks"].describe())

print("\nSeries with the most missing price weeks:")
display(
    price_summary
    .sort_values("missing_price_weeks", ascending=False)
    .head(10)
)

In [ ]:
# ============================================================
# CHECK WHETHER PRICE AVAILABILITY ENDS BEFORE THE DATASET ENDS
# ============================================================

last_dataset_week = calendar.loc[
    calendar["d"] == day_cols[-1], "wm_yr_wk"
].iloc[0]

print("Last sales day:", day_cols[-1])
print("Last dataset week:", last_dataset_week)

ended_early = price_summary[
    price_summary["last_price_week"] < last_dataset_week
]

print(
    "\nSeries whose price history ends before the final sales week:",
    len(ended_early),
    f"({len(ended_early) / len(price_summary):.2%})"
)

print("\nLast-price-week distribution:")
print(price_summary["last_price_week"].describe())

print("\nExamples ending earliest:")
display(
    price_summary
    .sort_values("last_week_position")
    .head(10)
)

In [ ]:
# ============================================================
# CALENDAR INTEGRITY AUDIT
# ============================================================

calendar_check = calendar.copy()
calendar_check["date"] = pd.to_datetime(calendar_check["date"])

print("Calendar rows:", len(calendar_check))
print("Date range:",
      calendar_check["date"].min(),
      "to",
      calendar_check["date"].max())

print("\nDuplicate dates:",
      calendar_check["date"].duplicated().sum())

print("Duplicate M5 day IDs:",
      calendar_check["d"].duplicated().sum())

# Check whether every date follows the previous date continuously.
date_gaps = calendar_check["date"].sort_values().diff().dt.days

print("Non-consecutive date gaps:",
      (date_gaps.dropna() != 1).sum())

# Check whether every sales day d_1 ... d_1941 exists in calendar.csv.
sales_days = set(day_cols)
calendar_days = set(calendar_check["d"])

missing_sales_days = sales_days - calendar_days

print("\nSales days missing from calendar:",
      len(missing_sales_days))

# Examine missing values column-by-column.
print("\nMissing values by calendar column:")
print(calendar_check.isna().sum())

# Examine possible values for event types and SNAP.
print("\nEvent type 1 values:")
print(calendar_check["event_type_1"].value_counts(dropna=False))

print("\nEvent type 2 values:")
print(calendar_check["event_type_2"].value_counts(dropna=False))

print("\nSNAP values:")
for col in ["snap_CA", "snap_TX", "snap_WI"]:
    print(f"\n{col}:")
    print(calendar_check[col].value_counts(dropna=False))

In [ ]:
# ============================================================
# DAILY DEMAND SUMMARY FOR CALENDAR/EVENT ANALYSIS
# ============================================================

# Total units sold across all item-store series for each M5 day.
daily_total_sales = sales[day_cols].sum(axis=0)

daily_demand = pd.DataFrame({
    "d": daily_total_sales.index,
    "total_sales": daily_total_sales.values
})

# Attach calendar information to each sales day.
daily_demand = daily_demand.merge(
    calendar_check,
    on="d",
    how="left",
    validate="one_to_one"
)

print("Rows:", len(daily_demand))
print("Missing calendar matches:", daily_demand["date"].isna().sum())

display(
    daily_demand[
        [
            "date",
            "d",
            "total_sales",
            "weekday",
            "month",
            "event_name_1",
            "event_type_1",
            "snap_CA",
            "snap_TX",
            "snap_WI",
        ]
    ].head(10)
)

In [ ]:
# ============================================================
# BASIC DEMAND SEASONALITY
# ============================================================

# Average total demand by weekday.
weekday_order = [
    "Monday", "Tuesday", "Wednesday", "Thursday",
    "Friday", "Saturday", "Sunday"
]

weekday_sales = (
    daily_demand.groupby("weekday")["total_sales"]
    .agg(["mean", "median", "std", "count"])
    .reindex(weekday_order)
)

print("AVERAGE DEMAND BY WEEKDAY:")
display(weekday_sales.round(2))


# Average total demand by month.
monthly_sales = (
    daily_demand.groupby("month")["total_sales"]
    .agg(["mean", "median", "std", "count"])
)

print("\nAVERAGE DEMAND BY MONTH:")
display(monthly_sales.round(2))


# Relative weekday effect.
overall_daily_mean = daily_demand["total_sales"].mean()

weekday_sales["vs_overall_%"] = (
    (weekday_sales["mean"] / overall_daily_mean - 1) * 100
)

print("\nWEEKDAY EFFECT RELATIVE TO OVERALL DAILY AVERAGE:")
display(weekday_sales[["mean", "vs_overall_%"]].round(2))

In [ ]:
# ============================================================
# EVENT / HOLIDAY DEMAND EFFECT
# ============================================================

event_rows = daily_demand[
    daily_demand["event_name_1"].notna()
].copy()

print("Number of event days:", len(event_rows))
print("Unique events:", event_rows["event_name_1"].nunique())

print("\nEvent types:")
print(event_rows["event_type_1"].value_counts())


# ------------------------------------------------------------
# Compare event days with ordinary days
# ------------------------------------------------------------

daily_demand["is_event"] = daily_demand["event_name_1"].notna()

event_comparison = (
    daily_demand.groupby("is_event")["total_sales"]
    .agg(["mean", "median", "std", "count"])
)

print("\nEVENT VS NON-EVENT DAYS:")
display(event_comparison.round(2))


# ------------------------------------------------------------
# Demand around events: -3 days through +3 days
# ------------------------------------------------------------

event_indices = daily_demand.index[daily_demand["is_event"]]

event_window_results = []

for offset in range(-3, 4):

    valid_indices = event_indices + offset

    # Keep only positions that actually exist in the dataset.
    valid_indices = valid_indices[
        (valid_indices >= 0) &
        (valid_indices < len(daily_demand))
    ]

    values = daily_demand.loc[valid_indices, "total_sales"]

    event_window_results.append({
        "days_from_event": offset,
        "mean_sales": values.mean(),
        "median_sales": values.median(),
        "count": len(values)
    })

event_window = pd.DataFrame(event_window_results)

print("\nDEMAND AROUND EVENTS:")
display(event_window.round(2))


# ------------------------------------------------------------
# Demand by event type
# ------------------------------------------------------------

event_type_sales = (
    event_rows.groupby("event_type_1")["total_sales"]
    .agg(["mean", "median", "std", "count"])
    .sort_values("mean", ascending=False)
)

print("\nDEMAND BY EVENT TYPE:")
display(event_type_sales.round(2))

In [ ]:
# ============================================================
# EVENT EFFECT AFTER CONTROLLING FOR WEEKDAY
# ============================================================

# Calculate normal average demand for each weekday.
weekday_baseline = (
    daily_demand.loc[~daily_demand["is_event"]]
    .groupby("weekday")["total_sales"]
    .mean()
)

# Expected sales based purely on weekday.
daily_demand["weekday_baseline"] = (
    daily_demand["weekday"].map(weekday_baseline)
)

# Percentage above/below the normal level for that weekday.
daily_demand["weekday_adjusted_pct"] = (
    (daily_demand["total_sales"] / daily_demand["weekday_baseline"] - 1)
    * 100
)

# ------------------------------------------------------------
# Examine -3 to +3 days around every event.
# ------------------------------------------------------------

event_indices = daily_demand.index[daily_demand["is_event"]]

adjusted_event_results = []

for offset in range(-3, 4):

    valid_indices = event_indices + offset

    valid_indices = valid_indices[
        (valid_indices >= 0) &
        (valid_indices < len(daily_demand))
    ]

    values = daily_demand.loc[
        valid_indices,
        "weekday_adjusted_pct"
    ]

    adjusted_event_results.append({
        "days_from_event": offset,
        "mean_adjusted_%": values.mean(),
        "median_adjusted_%": values.median(),
        "count": len(values)
    })

adjusted_event_window = pd.DataFrame(adjusted_event_results)

print("EVENT EFFECT AFTER CONTROLLING FOR WEEKDAY:")
display(adjusted_event_window.round(2))


# ------------------------------------------------------------
# Event-day effect by event type.
# ------------------------------------------------------------

event_type_adjusted = (
    daily_demand[daily_demand["is_event"]]
    .groupby("event_type_1")["weekday_adjusted_pct"]
    .agg(["mean", "median", "std", "count"])
    .sort_values("mean", ascending=False)
)

print("\nEVENT TYPE EFFECT AFTER CONTROLLING FOR WEEKDAY:")
display(event_type_adjusted.round(2))

In [ ]:
# ============================================================
# INDIVIDUAL EVENT EFFECTS
# ============================================================

event_effect_rows = []

# Analyze every unique primary event.
for event_name in daily_demand["event_name_1"].dropna().unique():

    event_indices = daily_demand.index[
        daily_demand["event_name_1"] == event_name
    ]

    row = {
        "event_name": event_name,
        "occurrences": len(event_indices)
    }

    # Examine two days before through one day after.
    for offset in [-2, -1, 0, 1]:

        valid_indices = event_indices + offset

        valid_indices = valid_indices[
            (valid_indices >= 0) &
            (valid_indices < len(daily_demand))
        ]

        values = daily_demand.loc[
            valid_indices,
            "weekday_adjusted_pct"
        ]

        # Use simple, predictable column names.
        column_name = {
            -2: "two_days_before_%",
            -1: "one_day_before_%",
             0: "event_day_%",
             1: "one_day_after_%"
        }[offset]

        row[column_name] = values.mean()

    event_effect_rows.append(row)


individual_event_effects = (
    pd.DataFrame(event_effect_rows)
    .sort_values("event_day_%", ascending=False)
    .reset_index(drop=True)
)

print("INDIVIDUAL EVENT EFFECTS")
print("(percentage above/below normal demand for the same weekday)")

display(individual_event_effects.round(2))

In [ ]:
# ============================================================
# EVENT EFFECTS BY PRODUCT CATEGORY
# ============================================================

category_daily_parts = []

for category in sales["cat_id"].unique():

    # Select every item-store series belonging to this category.
    category_mask = sales["cat_id"] == category

    # Aggregate sales across those series for each day.
    category_totals = sales.loc[category_mask, day_cols].sum(axis=0)

    temp = pd.DataFrame({
        "d": day_cols,
        "category": category,
        "category_sales": category_totals.values
    })

    category_daily_parts.append(temp)

category_daily = pd.concat(
    category_daily_parts,
    ignore_index=True
)

# Attach calendar information.
category_daily = category_daily.merge(
    calendar_check[
        ["d", "date", "weekday", "event_name_1", "event_type_1"]
    ],
    on="d",
    how="left",
    validate="many_to_one"
)

category_daily["is_event"] = (
    category_daily["event_name_1"].notna()
)


# ------------------------------------------------------------
# Create a normal weekday baseline separately for each category.
# ------------------------------------------------------------

category_weekday_baseline = (
    category_daily[~category_daily["is_event"]]
    .groupby(["category", "weekday"])["category_sales"]
    .mean()
    .rename("weekday_baseline")
    .reset_index()
)

category_daily = category_daily.merge(
    category_weekday_baseline,
    on=["category", "weekday"],
    how="left"
)

category_daily["weekday_adjusted_pct"] = (
    (
        category_daily["category_sales"]
        / category_daily["weekday_baseline"]
    ) - 1
) * 100


# ------------------------------------------------------------
# Analyze selected major events by category.
# ------------------------------------------------------------

major_events = [
    "SuperBowl",
    "Easter",
    "Mother's day",
    "Father's day",
    "IndependenceDay",
    "LaborDay",
    "Halloween",
    "Thanksgiving",
    "Christmas",
    "NewYear"
]

results = []

for category in category_daily["category"].unique():

    category_data = (
        category_daily[category_daily["category"] == category]
        .reset_index(drop=True)
    )

    for event in major_events:

        event_indices = category_data.index[
            category_data["event_name_1"] == event
        ]

        if len(event_indices) == 0:
            continue

        row = {
            "category": category,
            "event": event,
            "occurrences": len(event_indices)
        }

        for offset in [-2, -1, 0, 1]:

            valid_indices = event_indices + offset

            valid_indices = valid_indices[
                (valid_indices >= 0) &
                (valid_indices < len(category_data))
            ]

            values = category_data.loc[
                valid_indices,
                "weekday_adjusted_pct"
            ]

            column_name = {
                -2: "two_days_before_%",
                -1: "one_day_before_%",
                 0: "event_day_%",
                 1: "one_day_after_%"
            }[offset]

            row[column_name] = values.mean()

        results.append(row)


category_event_effects = pd.DataFrame(results)

display(
    category_event_effects
    .sort_values(["event", "category"])
    .round(2)
)

In [ ]:
# ============================================================
# SNAP EFFECT BY STATE AND PRODUCT CATEGORY
# ============================================================

snap_daily_parts = []

# Build daily demand for every state × category combination.
for state in sales["state_id"].unique():

    for category in sales["cat_id"].unique():

        mask = (
            (sales["state_id"] == state) &
            (sales["cat_id"] == category)
        )

        daily_totals = sales.loc[mask, day_cols].sum(axis=0)

        temp = pd.DataFrame({
            "d": day_cols,
            "state": state,
            "category": category,
            "sales": daily_totals.values
        })

        snap_daily_parts.append(temp)


snap_daily = pd.concat(
    snap_daily_parts,
    ignore_index=True
)


# ------------------------------------------------------------
# Attach calendar information.
# ------------------------------------------------------------

snap_daily = snap_daily.merge(
    calendar_check[
        [
            "d",
            "date",
            "weekday",
            "snap_CA",
            "snap_TX",
            "snap_WI"
        ]
    ],
    on="d",
    how="left",
    validate="many_to_one"
)


# ------------------------------------------------------------
# Each state should use ONLY its own SNAP indicator.
# ------------------------------------------------------------

snap_daily["snap_active"] = 0

for state in ["CA", "TX", "WI"]:

    state_mask = snap_daily["state"] == state

    snap_daily.loc[state_mask, "snap_active"] = (
        snap_daily.loc[state_mask, f"snap_{state}"]
    )


# ------------------------------------------------------------
# Build weekday baselines separately for every
# state × category combination.
# ------------------------------------------------------------

weekday_baseline = (
    snap_daily
    .groupby(["state", "category", "weekday"])["sales"]
    .mean()
    .rename("weekday_baseline")
    .reset_index()
)

snap_daily = snap_daily.merge(
    weekday_baseline,
    on=["state", "category", "weekday"],
    how="left"
)


# Percentage difference from normal demand for that weekday.
snap_daily["weekday_adjusted_pct"] = (
    (snap_daily["sales"] / snap_daily["weekday_baseline"]) - 1
) * 100


# ------------------------------------------------------------
# Compare SNAP vs non-SNAP days.
# ------------------------------------------------------------

snap_effect = (
    snap_daily
    .groupby(["state", "category", "snap_active"])
    ["weekday_adjusted_pct"]
    .agg(["mean", "median", "std", "count"])
    .reset_index()
)

print("SNAP EFFECT BY STATE AND CATEGORY")
print("(percentage relative to normal demand for the same weekday)")

display(
    snap_effect.round(2)
)

In [10]:
# ============================================================
# PRICE INTEGRITY AND VARIATION AUDIT
# ============================================================

print("Price rows:", f"{len(prices):,}")

print("\nMissing prices:")
print(prices["sell_price"].isna().sum())

print("\nZero or negative prices:")
print((prices["sell_price"] <= 0).sum())

print("\nOverall price distribution:")
print(prices["sell_price"].describe(
    percentiles=[0.01, 0.05, 0.25, 0.50, 0.75, 0.95, 0.99]
))


# ------------------------------------------------------------
# Check duplicate store-item-week records.
# ------------------------------------------------------------

duplicate_price_keys = prices.duplicated(
    subset=["store_id", "item_id", "wm_yr_wk"]
).sum()

print("\nDuplicate store-item-week price records:",
      duplicate_price_keys)


# ------------------------------------------------------------
# Measure price variation within each item-store series.
# ------------------------------------------------------------

series_price_stats = (
    prices.groupby(["store_id", "item_id"])["sell_price"]
    .agg(
        min_price="min",
        max_price="max",
        mean_price="mean",
        unique_prices="nunique"
    )
    .reset_index()
)

series_price_stats["price_range_pct"] = (
    (
        series_price_stats["max_price"]
        - series_price_stats["min_price"]
    )
    / series_price_stats["mean_price"]
) * 100


print("\nUnique-price-count distribution per item-store:")
print(series_price_stats["unique_prices"].describe())


print("\nSeries that never change price:",
      (series_price_stats["unique_prices"] == 1).sum(),
      f"({(series_price_stats['unique_prices'] == 1).mean():.2%})")


print("\nWithin-series price-range % distribution:")
print(
    series_price_stats["price_range_pct"].describe(
        percentiles=[0.25, 0.50, 0.75, 0.90, 0.95, 0.99]
    )
)


print("\nLargest price variation:")
display(
    series_price_stats
    .sort_values("price_range_pct", ascending=False)
    .head(10)
    .round(2)
)

Price rows: 6,841,121

Missing prices:
0

Zero or negative prices:
0

Overall price distribution:
count    6.841121e+06
mean     4.410952e+00
std      3.408814e+00
min      1.000000e-02
1%       4.800000e-01
5%       9.700000e-01
25%      2.180000e+00
50%      3.470000e+00
75%      5.840000e+00
95%      1.098000e+01
99%      1.792000e+01
max      1.073200e+02
Name: sell_price, dtype: float64

Duplicate store-item-week price records: 0

Unique-price-count distribution per item-store:
count    30490.000000
mean         2.769367
std          1.893627
min          1.000000
25%          1.000000
50%          2.000000
75%          4.000000
max         21.000000
Name: unique_prices, dtype: float64

Series that never change price: 8247 (27.05%)

Within-series price-range % distribution:
count    30490.000000
mean        14.439751
std         19.924300
min          0.000000
25%          0.000000
50%          8.801363
75%         19.259711
90%         38.680155
95%         55.826997
99%         

,store_id,item_id,min_price,max_price,mean_price,unique_prices,price_range_pct
30379,WI_3,HOUSEHOLD_2_406,3.26,107.32,13.12,4,793.39
14907,TX_1,HOUSEHOLD_2_178,3.00,44.36,7.37,4,561.52
19616,TX_3,FOODS_3_711,0.05,1.00,0.20,8,485.17
27174,WI_2,HOUSEHOLD_2_250,3.36,34.18,7.31,16,421.35
24125,WI_1,HOUSEHOLD_2_250,4.97,30.32,7.05,5,359.77
15194,TX_1,HOUSEHOLD_2_466,6.46,52.62,12.91,15,357.55
27330,WI_2,HOUSEHOLD_2_406,12.46,61.46,15.51,8,316.02
25629,WI_2,FOODS_3_625,2.50,11.25,3.03,4,289.22
30223,WI_3,HOUSEHOLD_2_250,6.96,23.62,7.32,5,227.58
24281,WI_1,HOUSEHOLD_2_406,9.97,35.88,12.64,5,205.06


In [11]:
# ============================================================
# INVESTIGATE EXTREME PRICE VARIATION
# ============================================================

extreme_series = (
    series_price_stats
    .sort_values("price_range_pct", ascending=False)
    .head(10)[["store_id", "item_id"]]
)

extreme_price_history = prices.merge(
    extreme_series,
    on=["store_id", "item_id"],
    how="inner"
)

extreme_price_history = extreme_price_history.sort_values(
    ["store_id", "item_id", "wm_yr_wk"]
)

# Show every distinct price and how many weeks it occurred.
extreme_price_levels = (
    extreme_price_history
    .groupby(["store_id", "item_id", "sell_price"])
    .agg(
        first_week=("wm_yr_wk", "min"),
        last_week=("wm_yr_wk", "max"),
        weeks=("wm_yr_wk", "count")
    )
    .reset_index()
    .sort_values(["store_id", "item_id", "sell_price"])
)

print("PRICE LEVELS FOR THE 10 MOST EXTREME SERIES:")
display(extreme_price_levels)


# Also inspect how common very large price ranges are.
print("\nNUMBER OF SERIES ABOVE DIFFERENT PRICE-RANGE THRESHOLDS:")

for threshold in [25, 50, 100, 200, 500]:
    count = (series_price_stats["price_range_pct"] > threshold).sum()

    print(
        f"> {threshold:>3}% range: "
        f"{count:>5,} series "
        f"({count / len(series_price_stats):.2%})"
    )

PRICE LEVELS FOR THE 10 MOST EXTREME SERIES:


,store_id,item_id,sell_price,first_week,last_week,weeks
0,TX_1,HOUSEHOLD_2_178,3.00,11215,11215,1
1,TX_1,HOUSEHOLD_2_178,6.96,11605,11621,17
2,TX_1,HOUSEHOLD_2_178,6.97,11110,11604,252
3,TX_1,HOUSEHOLD_2_178,44.36,11225,11227,3
4,TX_1,HOUSEHOLD_2_466,6.46,11604,11621,14
...,...,...,...,...,...,...
69,WI_3,HOUSEHOLD_2_250,23.62,11205,11208,4
70,WI_3,HOUSEHOLD_2_406,3.26,11242,11252,11
71,WI_3,HOUSEHOLD_2_406,12.46,11301,11621,175
72,WI_3,HOUSEHOLD_2_406,12.47,11102,11241,92



NUMBER OF SERIES ABOVE DIFFERENT PRICE-RANGE THRESHOLDS:
>  25% range: 5,482 series (17.98%)
>  50% range: 1,997 series (6.55%)
> 100% range:    83 series (0.27%)
> 200% range:    10 series (0.03%)
> 500% range:     2 series (0.01%)


In [12]:
# ============================================================
# PRICE CHANGE / DISCOUNT VS WEEKLY DEMAND
# ============================================================

# ------------------------------------------------------------
# 1. Convert daily sales into weekly sales.
# ------------------------------------------------------------

sales_id_cols = [
    "item_id",
    "dept_id",
    "cat_id",
    "store_id",
    "state_id"
]

# Map each sales day (d_1, d_2, ...) to its M5 week.
day_to_week = (
    calendar_check[
        calendar_check["d"].isin(day_cols)
    ][["d", "wm_yr_wk"]]
    .set_index("d")["wm_yr_wk"]
)

# Group the daily sales columns by week.
# Transposing here lets us aggregate d_1...d_1941 into weeks
# without first creating a huge daily long-format table.
weekly_sales_values = (
    sales[day_cols]
    .T
    .groupby(day_to_week)
    .sum()
    .T
)

weekly_sales_values.columns.name = None

# Convert the weekly matrix to long format.
weekly_sales = (
    weekly_sales_values
    .assign(
        item_id=sales["item_id"].values,
        dept_id=sales["dept_id"].values,
        cat_id=sales["cat_id"].values,
        store_id=sales["store_id"].values,
        state_id=sales["state_id"].values
    )
    .melt(
        id_vars=sales_id_cols,
        var_name="wm_yr_wk",
        value_name="weekly_sales"
    )
)


# ------------------------------------------------------------
# 2. Join weekly demand with weekly prices.
#
# Inner join intentionally keeps only weeks where the item
# has a recorded price — our previously frozen active-period
# definition.
# ------------------------------------------------------------

weekly_price_demand = weekly_sales.merge(
    prices,
    on=["store_id", "item_id", "wm_yr_wk"],
    how="inner",
    validate="one_to_one"
)

print("Active item-store-week observations:",
      f"{len(weekly_price_demand):,}")


# ------------------------------------------------------------
# 3. Calculate each series' typical price.
#
# Median is used because it is less distorted by rare extreme
# prices such as the $107 example we just investigated.
# ------------------------------------------------------------

typical_price = (
    weekly_price_demand
    .groupby(["store_id", "item_id"])["sell_price"]
    .median()
    .rename("typical_price")
    .reset_index()
)

weekly_price_demand = weekly_price_demand.merge(
    typical_price,
    on=["store_id", "item_id"],
    how="left",
    validate="many_to_one"
)

weekly_price_demand["price_vs_typical_pct"] = (
    (
        weekly_price_demand["sell_price"]
        / weekly_price_demand["typical_price"]
    ) - 1
) * 100


# ------------------------------------------------------------
# 4. Put price changes into interpretable buckets.
# ------------------------------------------------------------

weekly_price_demand["price_bucket"] = pd.cut(
    weekly_price_demand["price_vs_typical_pct"],
    bins=[
        -np.inf,
        -20,
        -10,
        -5,
        5,
        10,
        20,
        np.inf
    ],
    labels=[
        ">20% cheaper",
        "10-20% cheaper",
        "5-10% cheaper",
        "Typical ±5%",
        "5-10% higher",
        "10-20% higher",
        ">20% higher"
    ]
)


# ------------------------------------------------------------
# 5. Compare weekly demand against each series' own
# typical weekly demand.
# ------------------------------------------------------------

typical_weekly_demand = (
    weekly_price_demand
    .groupby(["store_id", "item_id"])["weekly_sales"]
    .median()
    .rename("typical_weekly_sales")
    .reset_index()
)

weekly_price_demand = weekly_price_demand.merge(
    typical_weekly_demand,
    on=["store_id", "item_id"],
    how="left",
    validate="many_to_one"
)

# Avoid percentage division by zero for extremely intermittent
# products whose median weekly sales are zero.
valid_price_analysis = weekly_price_demand[
    weekly_price_demand["typical_weekly_sales"] > 0
].copy()

valid_price_analysis["demand_vs_typical_pct"] = (
    (
        valid_price_analysis["weekly_sales"]
        / valid_price_analysis["typical_weekly_sales"]
    ) - 1
) * 100


# ------------------------------------------------------------
# 6. Summarize price-demand relationship.
# ------------------------------------------------------------

price_demand_summary = (
    valid_price_analysis
    .groupby(
        ["cat_id", "price_bucket"],
        observed=True
    )["demand_vs_typical_pct"]
    .agg(["mean", "median", "count"])
    .reset_index()
)

print("\nPRICE VS DEMAND BY CATEGORY")
print("Demand change is relative to each item's typical weekly demand.")

display(
    price_demand_summary.round(2)
)

Active item-store-week observations: 6,719,161

PRICE VS DEMAND BY CATEGORY
Demand change is relative to each item's typical weekly demand.


,cat_id,price_bucket,mean,median,count
0,FOODS,>20% cheaper,56.35,8.79,33423
1,FOODS,10-20% cheaper,68.18,25.00,119176
2,FOODS,5-10% cheaper,39.43,13.33,163159
3,FOODS,Typical ±5%,18.90,0.00,2285584
4,FOODS,5-10% higher,16.80,-4.76,181643
5,FOODS,10-20% higher,16.44,-21.05,123710
6,FOODS,>20% higher,-9.57,-30.77,19039
7,HOBBIES,>20% cheaper,40.06,0.00,5568
8,HOBBIES,10-20% cheaper,47.35,0.00,38556
9,HOBBIES,5-10% cheaper,36.74,0.00,53364


In [ ]:
# ============================================================
# SALES SPIKE / OUTLIER AUDIT
# ============================================================

# Use the active weekly dataset we already created in Cell 22.
# This avoids treating pre-availability periods as zero demand.

weekly_series_stats = (
    weekly_price_demand
    .groupby(["store_id", "item_id"])["weekly_sales"]
    .agg(
        mean_weekly_sales="mean",
        median_weekly_sales="median",
        std_weekly_sales="std",
        max_weekly_sales="max"
    )
    .reset_index()
)

# ------------------------------------------------------------
# Global distribution of active weekly demand
# ------------------------------------------------------------

print("ACTIVE WEEKLY SALES DISTRIBUTION:")

print(
    weekly_price_demand["weekly_sales"].describe(
        percentiles=[
            0.50, 0.75, 0.90, 0.95,
            0.99, 0.995, 0.999
        ]
    )
)


# ------------------------------------------------------------
# How extreme is each series' maximum compared with its
# own normal demand?
#
# We use the mean here because many intermittent products
# have median weekly demand = 0.
# ------------------------------------------------------------

weekly_series_stats["max_vs_mean"] = (
    weekly_series_stats["max_weekly_sales"]
    / weekly_series_stats["mean_weekly_sales"].replace(0, np.nan)
)


print("\nMAX / MEAN RATIO ACROSS SERIES:")

print(
    weekly_series_stats["max_vs_mean"].describe(
        percentiles=[0.50, 0.75, 0.90, 0.95, 0.99, 0.999]
    )
)


# ------------------------------------------------------------
# Show the largest absolute weekly sales observations.
# ------------------------------------------------------------

largest_weeks = (
    weekly_price_demand
    .sort_values("weekly_sales", ascending=False)
    [
        [
            "store_id",
            "item_id",
            "cat_id",
            "state_id",
            "wm_yr_wk",
            "weekly_sales",
            "sell_price",
            "price_vs_typical_pct"
        ]
    ]
    .head(20)
)

print("\n20 LARGEST ITEM-STORE WEEKLY SALES OBSERVATIONS:")

display(
    largest_weeks.round(2)
)


# ------------------------------------------------------------
# Show series with unusually extreme peaks relative to
# their own average.
# ------------------------------------------------------------

print("\nSERIES WITH LARGEST PEAK RELATIVE TO THEIR OWN MEAN:")

display(
    weekly_series_stats
    .sort_values("max_vs_mean", ascending=False)
    .head(20)
    .round(2)
)

In [ ]:
# ============================================================
# SERIES QUALITY / INTERMITTENCY AUDIT
# ============================================================

# We use weekly_price_demand because it already contains only
# active periods according to our frozen first-price rule.

series_quality = (
    weekly_price_demand
    .groupby(["store_id", "item_id", "cat_id", "state_id"])
    .agg(
        active_weeks=("weekly_sales", "size"),
        total_sales=("weekly_sales", "sum"),
        mean_weekly_sales=("weekly_sales", "mean"),
        median_weekly_sales=("weekly_sales", "median"),
        std_weekly_sales=("weekly_sales", "std"),
        zero_weeks=("weekly_sales", lambda x: (x == 0).sum()),
        positive_weeks=("weekly_sales", lambda x: (x > 0).sum())
    )
    .reset_index()
)


# ------------------------------------------------------------
# Intermittency:
# What percentage of active weeks had zero demand?
# ------------------------------------------------------------

series_quality["zero_week_pct"] = (
    series_quality["zero_weeks"]
    / series_quality["active_weeks"]
) * 100


# ------------------------------------------------------------
# Coefficient of variation (CV):
# standard deviation relative to average demand.
#
# Higher CV = demand is more volatile relative to its scale.
# ------------------------------------------------------------

series_quality["cv"] = (
    series_quality["std_weekly_sales"]
    / series_quality["mean_weekly_sales"].replace(0, np.nan)
)


print("ACTIVE HISTORY LENGTH (WEEKS):")

print(
    series_quality["active_weeks"].describe(
        percentiles=[0.10, 0.25, 0.50, 0.75, 0.90, 0.95, 0.99]
    )
)


print("\nZERO-DEMAND WEEK % PER SERIES:")

print(
    series_quality["zero_week_pct"].describe(
        percentiles=[0.10, 0.25, 0.50, 0.75, 0.90, 0.95, 0.99]
    )
)


print("\nDEMAND VOLATILITY (CV):")

print(
    series_quality["cv"].describe(
        percentiles=[0.10, 0.25, 0.50, 0.75, 0.90, 0.95, 0.99]
    )
)


# ------------------------------------------------------------
# Category-level comparison
# ------------------------------------------------------------

category_quality = (
    series_quality
    .groupby("cat_id")
    .agg(
        series_count=("item_id", "size"),
        median_active_weeks=("active_weeks", "median"),
        median_weekly_sales=("mean_weekly_sales", "median"),
        median_zero_week_pct=("zero_week_pct", "median"),
        median_cv=("cv", "median")
    )
    .reset_index()
)


print("\nSERIES QUALITY BY CATEGORY:")

display(
    category_quality.round(2)
)


# ------------------------------------------------------------
# How many series have relatively short histories?
# ------------------------------------------------------------

print("\nSHORT-HISTORY SERIES:")

for weeks in [8, 16, 26, 52, 104]:
    count = (series_quality["active_weeks"] < weeks).sum()

    print(
        f"< {weeks:>3} active weeks: "
        f"{count:>5,} series "
        f"({count / len(series_quality):.2%})"
    )


# ------------------------------------------------------------
# Extremely intermittent series
# ------------------------------------------------------------

print("\nHIGHLY INTERMITTENT SERIES:")

for threshold in [50, 75, 90, 95]:
    count = (
        series_quality["zero_week_pct"] >= threshold
    ).sum()

    print(
        f">= {threshold}% zero weeks: "
        f"{count:>5,} series "
        f"({count / len(series_quality):.2%})"
    )

In [1]:
# ============================================================
# CELL 25 — HIERARCHY + JOIN INTEGRITY
# ============================================================

# ------------------------------------------------------------
# 1. Verify the product/store hierarchy is consistent.
#
# Each item should always belong to the same department/category.
# Each store should always belong to the same state.
# ------------------------------------------------------------

item_hierarchy = (
    sales.groupby("item_id")
    .agg(
        departments=("dept_id", "nunique"),
        categories=("cat_id", "nunique")
    )
)

store_hierarchy = (
    sales.groupby("store_id")
    .agg(
        states=("state_id", "nunique")
    )
)

print("ITEM HIERARCHY PROBLEMS:")
print(
    "Items linked to multiple departments:",
    (item_hierarchy["departments"] > 1).sum()
)

print(
    "Items linked to multiple categories:",
    (item_hierarchy["categories"] > 1).sum()
)

print("\nSTORE HIERARCHY PROBLEMS:")
print(
    "Stores linked to multiple states:",
    (store_hierarchy["states"] > 1).sum()
)


# ------------------------------------------------------------
# 2. Check expected hierarchy sizes.
# ------------------------------------------------------------

print("\nHIERARCHY COUNTS:")

print("States:", sales["state_id"].nunique())
print("Stores:", sales["store_id"].nunique())
print("Categories:", sales["cat_id"].nunique())
print("Departments:", sales["dept_id"].nunique())
print("Items:", sales["item_id"].nunique())
print("Item-store series:", len(sales))


# ------------------------------------------------------------
# 3. Verify calendar coverage for every sales day.
# ------------------------------------------------------------

sales_day_set = set(day_cols)
calendar_day_set = set(calendar_check["d"])

missing_calendar_days = sales_day_set - calendar_day_set

print("\nCALENDAR JOIN CHECK:")
print(
    "Sales days missing from calendar:",
    len(missing_calendar_days)
)


# ------------------------------------------------------------
# 4. Verify the weekly sales-price join.
#
# weekly_sales contains every item-store-week combination.
# weekly_price_demand contains only combinations with a
# recorded price (our active-period definition).
# ------------------------------------------------------------

print("\nPRICE JOIN CHECK:")

print(
    "Weekly rows before price join:",
    f"{len(weekly_sales):,}"
)

print(
    "Active rows after price join:",
    f"{len(weekly_price_demand):,}"
)

missing_price_rows = (
    len(weekly_sales) - len(weekly_price_demand)
)

print(
    "Rows without recorded price:",
    f"{missing_price_rows:,}"
)

print(
    "Percent excluded as pre-availability:",
    f"{missing_price_rows / len(weekly_sales):.2%}"
)


# ------------------------------------------------------------
# 5. Ensure the price join did NOT duplicate observations.
# ------------------------------------------------------------

duplicate_active_keys = weekly_price_demand.duplicated(
    subset=["store_id", "item_id", "wm_yr_wk"]
).sum()

print(
    "\nDuplicate item-store-week rows after join:",
    duplicate_active_keys
)


# ------------------------------------------------------------
# 6. Check that all 30,490 series survived.
# ------------------------------------------------------------

active_series_count = (
    weekly_price_demand[
        ["store_id", "item_id"]
    ]
    .drop_duplicates()
    .shape[0]
)

print(
    "\nSeries remaining after active-period join:",
    f"{active_series_count:,}"
)

print(
    "Original series:",
    f"{len(sales):,}"
)

NameError: name 'sales' is not defined

## 11. Feature Availability and Leakage Contract

Before model training, every feature is classified according to whether it would
actually be available at the time a forecast is generated.

### Static features
These do not change during the forecast horizon:
- item_id
- dept_id
- cat_id
- store_id
- state_id

### Known-future features
These can legitimately be known for future dates:
- date
- weekday / wday
- month
- year
- scheduled event identity and type
- secondary event identity and type
- event proximity features
- the SNAP schedule corresponding to the store's state

### Past-observed features
These may be used only when calculated from information available on or before
the forecast origin:
- historical unit sales
- lag features
- rolling demand statistics
- historical sell prices
- historical price changes
- relative-price features
- product age / time since activation

All lag and rolling features must be shifted appropriately so that future target
values cannot enter model inputs.

### Future price and promotion rule
Future realized prices are not assumed to be known automatically.

A future price or promotion may be used as a known-future feature only when it
represents information explicitly supplied or scheduled by the business at
forecast time.

Historical M5 prices may be used for past-observed price features. Evaluation
must not give the model future realized prices unless that experiment is
explicitly labeled as a planned-price scenario.

### Unknown-future information
The following must never be exposed to the model at forecast time:
- actual future unit sales
- lag or rolling features containing future sales
- unplanned future price changes
- any feature constructed using information unavailable at the forecast origin

This contract applies to both TFT and LightGBM so that their evaluation remains
leakage-safe and operationally realistic.

In [14]:

import numpy as np
import pandas as pd

# STEP 1: Arrange weeks in actual calendar order.
# This avoids assuming M5 week IDs are consecutive numbers.
calendar_weeks = (
    calendar.assign(date=pd.to_datetime(calendar["date"]))
    .groupby("wm_yr_wk")["date"]
    .min()
    .sort_values()
)

week_order = pd.Series(
    np.arange(len(calendar_weeks)),
    index=calendar_weeks.index
)

# STEP 2: Create a separate copy of the price data.
# We will NOT change the original prices DataFrame.
price_audit = prices.copy()

price_audit["week_order"] = (
    price_audit["wm_yr_wk"].map(week_order)
)

assert price_audit["week_order"].notna().all()

price_audit = (
    price_audit
    .sort_values(["store_id", "item_id", "week_order"])
    .reset_index(drop=True)
)

# STEP 3: Give each product-store history a group number.
price_audit["series_id"] = (
    price_audit.groupby(
        ["store_id", "item_id"],
        sort=False
    ).ngroup()
)

# STEP 4: Find the previous week's observed price.
# shift(1) prevents the current price from influencing
# its own historical reference.
previous_prices = (
    price_audit.groupby("series_id")["sell_price"]
    .shift(1)
)

# STEP 5: Calculate the median of the previous
# 8 observed prices, requiring at least 4.
price_audit["reference_price_asof"] = (
    previous_prices
    .groupby(price_audit["series_id"])
    .rolling(window=8, min_periods=4)
    .median()
    .reset_index(level=0, drop=True)
)

# STEP 6: Compare current price with the recent reference.
price_audit["price_ratio_asof"] = (
    price_audit["sell_price"]
    / price_audit["reference_price_asof"]
)

price_audit["log_price_ratio_asof"] = np.log(
    price_audit["price_ratio_asof"]
)

# STEP 7: Flag the very low prices.
price_audit["extreme_low_price_flag"] = (
    price_audit["sell_price"] <= 0.10
)

# Check that our earlier count is reproduced.
assert price_audit["extreme_low_price_flag"].sum() == 331

print("PRICE AUDIT — FIRST CHECK")
print("Total weekly price records:", len(price_audit))

print(
    "Prices at or below $0.10:",
    int(price_audit["extreme_low_price_flag"].sum())
)

print(
    "Records with enough previous price history:",
    int(price_audit["reference_price_asof"].notna().sum())
)

print("\nEXAMPLES OF VERY LOW PRICES:")

display(
    price_audit.loc[
        price_audit["extreme_low_price_flag"],
        [
            "item_id",
            "store_id",
            "wm_yr_wk",
            "sell_price",
            "reference_price_asof",
            "price_ratio_asof"
        ]
    ].head(10).round(4)
)


PRICE AUDIT — FIRST CHECK
Total weekly price records: 6841121
Prices at or below $0.10: 331
Records with enough previous price history: 6719161

EXAMPLES OF VERY LOW PRICES:


,item_id,store_id,wm_yr_wk,sell_price,reference_price_asof,price_ratio_asof
381654,HOBBIES_1_234,CA_1,11304,0.1,0.3,0.3333
381655,HOBBIES_1_234,CA_1,11305,0.1,0.3,0.3333
381656,HOBBIES_1_234,CA_1,11306,0.1,0.3,0.3333
381657,HOBBIES_1_234,CA_1,11307,0.1,0.3,0.3333
381658,HOBBIES_1_234,CA_1,11308,0.1,0.2,0.5000
381659,HOBBIES_1_234,CA_1,11309,0.1,0.1,1.0000
381660,HOBBIES_1_234,CA_1,11310,0.1,0.1,1.0000
381661,HOBBIES_1_234,CA_1,11311,0.1,0.1,1.0000
381662,HOBBIES_1_234,CA_1,11312,0.1,0.1,1.0000
381663,HOBBIES_1_234,CA_1,11313,0.1,0.1,1.0000


In [15]:

# STEP 1: Look at the previous week's price for each product-store pair.
price_audit["previous_price"] = (
    price_audit.groupby("series_id")["sell_price"].shift(1)
)

# STEP 2: Find large changes that also differ substantially
# from the recent historical reference.
price_audit["change_from_previous"] = (
    price_audit["sell_price"] / price_audit["previous_price"]
)

large_drop = (
    (price_audit["change_from_previous"] <= 0.50)
    & (price_audit["price_ratio_asof"] <= 0.50)
)

large_increase = (
    (price_audit["change_from_previous"] >= 2.00)
    & (price_audit["price_ratio_asof"] >= 2.00)
)

price_audit["large_price_event_flag"] = (
    large_drop | large_increase
)

# STEP 3: Create a smaller table containing only event starts.
events = price_audit.loc[
    price_audit["large_price_event_flag"]
].copy()

# We compare future prices against the original reference.
# These future observations are used ONLY to label the audit,
# not to create forecasting features.
events["price_event_class"] = "Other/unresolved"

price_values = price_audit["sell_price"].to_numpy()
series_ids = price_audit["series_id"].to_numpy()
week_numbers = price_audit["week_order"].to_numpy()

# STEP 4: Examine the next 4 consecutive calendar weeks.
for index, event in events.iterrows():

    reference = event["reference_price_asof"]
    event_week = event["week_order"]
    event_series = event["series_id"]

    future_prices = []

    for offset in range(1, 5):

        next_index = index + offset

        if next_index >= len(price_audit):
            break

        # Stop if we leave the series or skip a calendar week.
        if (
            series_ids[next_index] != event_series
            or week_numbers[next_index] != event_week + offset
        ):
            break

        future_prices.append(price_values[next_index])

    # A return means the price comes back within 5%
    # of the reference price.
    returned = any(
        abs(future_price / reference - 1) <= 0.05
        for future_price in future_prices
    )

    is_drop = event["change_from_previous"] <= 0.50

    # A lasting change must persist through the next 4 weeks,
    # making 5 weeks including the event week.
    if returned:
        events.at[index, "price_event_class"] = (
            "Temporary dip" if is_drop else "Isolated spike"
        )

    elif len(future_prices) == 4:

        future_ratios = [
            future_price / reference
            for future_price in future_prices
        ]

        if is_drop and all(ratio <= 0.80 for ratio in future_ratios):
            events.at[index, "price_event_class"] = "Level shift"

        elif (not is_drop) and all(
            ratio >= 1.20 for ratio in future_ratios
        ):
            events.at[index, "price_event_class"] = "Level shift"

# STEP 5: Show results.
print("TOTAL LARGE PRICE EVENTS:", len(events))

print("\nEVENT CLASSIFICATION:")
display(
    events["price_event_class"]
    .value_counts()
    .to_frame("events")
)

print("\nLOW-PRICE RECORDS AMONG THE EVENTS:")
display(
    events.loc[
        events["extreme_low_price_flag"],
        "price_event_class"
    ].value_counts().to_frame("records")
)

print("\nEXAMPLE EVENTS:")
display(
    events[
        [
            "item_id",
            "store_id",
            "sell_price",
            "reference_price_asof",
            "change_from_previous",
            "price_event_class"
        ]
    ].head(10).round(3)
)


TOTAL LARGE PRICE EVENTS: 2366

EVENT CLASSIFICATION:


,events
price_event_class,
Temporary dip,1888
Level shift,336
Other/unresolved,113
Isolated spike,29



LOW-PRICE RECORDS AMONG THE EVENTS:


,records
price_event_class,
Temporary dip,208
Level shift,6
Other/unresolved,4



EXAMPLE EVENTS:


,item_id,store_id,sell_price,reference_price_asof,change_from_previous,price_event_class
37772,FOODS_1_158,CA_1,1.34,4.68,0.286,Temporary dip
92082,FOODS_2_177,CA_1,1.00,3.33,0.300,Temporary dip
92116,FOODS_2_177,CA_1,1.49,3.33,0.447,Temporary dip
92124,FOODS_2_177,CA_1,1.00,3.33,0.300,Temporary dip
99652,FOODS_2_211,CA_1,0.99,2.28,0.434,Temporary dip
355351,HOBBIES_1_112,CA_1,1.00,2.68,0.373,Temporary dip
381654,HOBBIES_1_234,CA_1,0.10,0.30,0.333,Level shift
381673,HOBBIES_1_234,CA_1,0.30,0.10,3.000,Level shift
395639,HOBBIES_1_295,CA_1,0.20,0.46,0.435,Temporary dip
405179,HOBBIES_1_333,CA_1,0.97,4.97,0.195,Temporary dip


In [16]:

# STEP 1: Find the previous recorded calendar-week position
# for each product-store combination.
price_audit["previous_week_order"] = (
    price_audit.groupby("series_id")["week_order"]
    .shift(1)
)

# STEP 2: Calculate how many calendar weeks separate
# the current price record from the preceding one.
price_audit["gap_weeks"] = (
    price_audit["week_order"]
    - price_audit["previous_week_order"]
)

# STEP 3: Count only records that actually have
# a preceding price observation.
has_previous_price = (
    price_audit["previous_week_order"].notna()
)

consecutive_week = price_audit["gap_weeks"].eq(1)

print("PRICE HISTORY CONTINUITY CHECK")

print(
    "Records with a preceding price:",
    int(has_previous_price.sum())
)

print(
    "Consecutive-week price records:",
    int((has_previous_price & consecutive_week).sum())
)

print(
    "Records with gaps greater than one week:",
    int((price_audit["gap_weeks"] > 1).sum())
)

# STEP 4: Check how many of our previously identified
# large price events crossed missing calendar weeks.
print("\nPREVIOUSLY IDENTIFIED LARGE PRICE EVENTS")

print(
    "Total:",
    int(price_audit["large_price_event_flag"].sum())
)

print(
    "Events with a genuine consecutive preceding week:",
    int(
        (
            price_audit["large_price_event_flag"]
            & consecutive_week
        ).sum()
    )
)

print(
    "Events crossing a missing-week gap:",
    int(
        (
            price_audit["large_price_event_flag"]
            & (price_audit["gap_weeks"] > 1)
        ).sum()
    )
)


PRICE HISTORY CONTINUITY CHECK
Records with a preceding price: 6810631
Consecutive-week price records: 6810631
Records with gaps greater than one week: 0

PREVIOUSLY IDENTIFIED LARGE PRICE EVENTS
Total: 2366
Events with a genuine consecutive preceding week: 2366
Events crossing a missing-week gap: 0


In [17]:

# ============================================================
# CORRECT PRICE EVENT CLASSIFICATION
# ============================================================

# Keep our original results so we can compare before/after.
events_corrected = events.copy()

price_values = price_audit["sell_price"].to_numpy()
series_ids = price_audit["series_id"].to_numpy()
week_numbers = price_audit["week_order"].to_numpy()

event_positions = events_corrected.index.to_numpy()
event_references = events_corrected[
    "reference_price_asof"
].to_numpy()

# Identify whether each event was a price drop or increase.
is_drop = (
    events_corrected["change_from_previous"].to_numpy()
    <= 0.50
)

# Store the next four prices for each event.
# Missing future observations remain NaN.
future_prices = np.full(
    (len(events_corrected), 4),
    np.nan
)

for offset in range(1, 5):

    next_positions = event_positions + offset

    valid = next_positions < len(price_audit)

    safe_positions = np.minimum(
        next_positions,
        len(price_audit) - 1
    )

    valid &= (
        series_ids[safe_positions]
        == series_ids[event_positions]
    )

    valid &= (
        week_numbers[safe_positions]
        == week_numbers[event_positions] + offset
    )

    future_prices[valid, offset - 1] = (
        price_values[next_positions[valid]]
    )

# Compare every future price against the ORIGINAL
# reference saved when the event first happened.
future_ratios = (
    future_prices / event_references[:, None]
)

# A price has returned if it gets within 5% of
# its original reference during the next four weeks.
returned = (
    (future_ratios >= 0.95)
    & (future_ratios <= 1.05)
).any(axis=1)

# All four following weeks must be available before
# we classify a persistent change.
complete_followup = (
    np.isfinite(future_prices).all(axis=1)
)

# Check whether prices remained meaningfully changed.
persistent_drop = (
    is_drop
    & (future_ratios <= 0.80).all(axis=1)
)

persistent_increase = (
    ~is_drop
    & (future_ratios >= 1.20).all(axis=1)
)

persistent = (
    complete_followup
    & (persistent_drop | persistent_increase)
)

# Classify using a clear priority:
# 1. Reversion, 2. Persistent change,
# 3. Incomplete follow-up, 4. Other.
new_classes = np.full(
    len(events_corrected),
    "Other/unresolved",
    dtype=object
)

new_classes[~complete_followup] = "Incomplete follow-up"

new_classes[persistent] = "Persistent change"

new_classes[returned & is_drop] = "Temporary dip"

new_classes[returned & ~is_drop] = "Isolated spike"

events_corrected["corrected_event_class"] = new_classes

# Compare our old and improved classifications.
print("OLD VS CORRECTED CLASSIFICATION")

comparison = pd.crosstab(
    events_corrected["price_event_class"],
    events_corrected["corrected_event_class"],
    rownames=["Old classification"],
    colnames=["Corrected classification"]
)

display(comparison)

print("\nCORRECTED EVENT COUNTS")

display(
    events_corrected["corrected_event_class"]
    .value_counts()
    .to_frame("events")
)

print("\nCHECKS")

print("Original events:", len(events))
print("Corrected events:", len(events_corrected))

assert len(events_corrected) == 2366

assert (
    events_corrected["corrected_event_class"]
    .notna().all()
)

print("All events classified successfully.")


OLD VS CORRECTED CLASSIFICATION


Corrected classification,Incomplete follow-up,Isolated spike,Other/unresolved,Persistent change,Temporary dip
Old classification,,,,,
Isolated spike,0,29,0,0,0
Level shift,0,0,0,336,0
Other/unresolved,12,0,101,0,0
Temporary dip,0,0,0,0,1888



CORRECTED EVENT COUNTS


,events
corrected_event_class,
Temporary dip,1888
Persistent change,336
Other/unresolved,101
Isolated spike,29
Incomplete follow-up,12



CHECKS
Original events: 2366
Corrected events: 2366
All events classified successfully.


In [18]:

# ============================================================
# GROUP EXTREMELY LOW PRICES INTO CONTINUOUS EPISODES
# ============================================================

# STEP 1: Select the 331 low-price records.
low_price_rows = price_audit.loc[
    price_audit["extreme_low_price_flag"]
].copy()

# STEP 2: Identify when a new low-price episode starts.
# A new episode begins when:
# - We encounter a different product-store combination, OR
# - The previous low-price week is not directly consecutive.

previous_low_series = low_price_rows["series_id"].shift(1)
previous_low_week = low_price_rows["week_order"].shift(1)

new_episode = (
    low_price_rows["series_id"].ne(previous_low_series)
    | low_price_rows["week_order"].sub(previous_low_week).ne(1)
)

# Give each continuous episode a unique number.
low_price_rows["low_price_episode_id"] = (
    new_episode.cumsum()
)

# STEP 3: Summarize each episode.
low_price_episodes = (
    low_price_rows
    .groupby("low_price_episode_id")
    .agg(
        item_id=("item_id", "first"),
        store_id=("store_id", "first"),
        start_week=("wm_yr_wk", "first"),
        end_week=("wm_yr_wk", "last"),
        start_week_order=("week_order", "first"),
        end_week_order=("week_order", "last"),
        episode_weeks=("week_order", "size"),
        minimum_price=("sell_price", "min"),
        maximum_price=("sell_price", "max")
    )
    .reset_index()
)

# STEP 4: Confirm that ALL 331 records are included.
assert len(low_price_rows) == 331

assert low_price_episodes["episode_weeks"].sum() == 331

print("LOW-PRICE EPISODE SUMMARY")

print("Total low-price records:", len(low_price_rows))
print("Separate low-price episodes:", len(low_price_episodes))
print(
    "Total weeks across episodes:",
    int(low_price_episodes["episode_weeks"].sum())
)

print("\nEPISODE LENGTH DISTRIBUTION")

display(
    low_price_episodes["episode_weeks"]
    .value_counts()
    .sort_index()
    .to_frame("episodes")
)

print("\nLONGEST LOW-PRICE EPISODES")

display(
    low_price_episodes[
        [
            "item_id",
            "store_id",
            "episode_weeks",
            "minimum_price",
            "maximum_price"
        ]
    ]
    .sort_values("episode_weeks", ascending=False)
    .head(10)
)


LOW-PRICE EPISODE SUMMARY
Total low-price records: 331
Separate low-price episodes: 224
Total weeks across episodes: 331

EPISODE LENGTH DISTRIBUTION


,episodes
episode_weeks,
1,185
2,25
3,7
4,1
6,1
7,1
9,1
12,1
18,1



LONGEST LOW-PRICE EPISODES


,item_id,store_id,episode_weeks,minimum_price,maximum_price
0,HOBBIES_1_234,CA_1,19,0.10,0.10
75,FOODS_3_454,TX_2,18,0.10,0.10
98,FOODS_3_669,TX_2,12,0.10,0.10
192,FOODS_3_711,TX_3,9,0.08,0.08
126,FOODS_3_070,TX_3,7,0.09,0.09
221,HOBBIES_2_128,WI_2,6,0.07,0.07
133,FOODS_3_234,TX_3,4,0.10,0.10
90,FOODS_3_547,TX_2,3,0.08,0.08
101,FOODS_3_711,TX_2,3,0.05,0.05
151,FOODS_3_454,TX_3,3,0.05,0.05


In [8]:

# ============================================================
# FINAL PRICE AUDIT — STEP 1: VERIFICATION
# ============================================================

# Check that all observed price histories are consecutive
# using actual calendar-week positions.
assert (
    price_audit.loc[
        price_audit["previous_week_order"].notna(),
        "gap_weeks"
    ] == 1
).all()

assert (
    price_audit["previous_week_order"].notna().sum()
    == len(price_audit) - price_audit["series_id"].nunique()
)

print("Calendar-week continuity: PASSED")

# Check that incomplete-follow-up events occur only
# within the final 4 calendar weeks.
last_week = price_audit["week_order"].max()

incomplete_events = events_corrected[
    events_corrected["corrected_event_class"]
    == "Incomplete follow-up"
]

assert (
    incomplete_events["week_order"] > last_week - 4
).all()

assert len(incomplete_events) == 12

print("Incomplete follow-up boundary check: PASSED")
print("Incomplete events:", len(incomplete_events))
print("All verification checks passed.")


Calendar-week continuity: PASSED
Incomplete follow-up boundary check: PASSED
Incomplete events: 12
All verification checks passed.


In [19]:

# ============================================================
# PRICE AUDIT — STEP 2: EPISODE ENTRY AND EXIT
# ============================================================

# Each low-price episode already has a unique ID.
# Find the first and last row belonging to every episode.
episode_boundaries = (
    low_price_rows
    .groupby("low_price_episode_id")
    .agg(
        first_row=("week_order", lambda x: x.index[0]),
        last_row=("week_order", lambda x: x.index[-1])
    )
    .reset_index()
)

# Attach these boundaries to our existing episode table.
low_price_episodes = (
    low_price_episodes
    .drop(columns=["first_row", "last_row",
                   "entry_type", "exit_type"], errors="ignore")
    .merge(
        episode_boundaries,
        on="low_price_episode_id",
        how="left",
        validate="one_to_one"
    )
)

# Arrays let us inspect the neighboring price records.
series_values = price_audit["series_id"].to_numpy()
price_values = price_audit["sell_price"].to_numpy()
week_values = price_audit["week_order"].to_numpy()

entry_results = []
exit_results = []

for episode in low_price_episodes.itertuples():

    first_row = int(episode.first_row)
    last_row = int(episode.last_row)

    # ------------------------------------------
    # ENTRY: How did the low-price episode begin?
    # ------------------------------------------

    if (
        first_row == 0
        or series_values[first_row - 1] != series_values[first_row]
    ):
        entry_type = "Series start"

    elif (
        price_audit.iloc[first_row]["large_price_event_flag"]
        and price_audit.iloc[first_row]["change_from_previous"] <= 0.50
    ):
        entry_type = "Large drop event"

    else:
        entry_type = "Other entry"

    entry_results.append(entry_type)

    # ------------------------------------------
    # EXIT: What happened immediately afterward?
    # ------------------------------------------

    next_row = last_row + 1

    if (
        next_row >= len(price_audit)
        or series_values[next_row] != series_values[last_row]
    ):
        exit_type = "Series end"

    elif (
        week_values[next_row] == week_values[last_row] + 1
        and price_audit.iloc[next_row]["large_price_event_flag"]
        and price_audit.iloc[next_row]["change_from_previous"] >= 2.0
    ):
        exit_type = "Large increase event"

    else:
        exit_type = "Other exit"

    exit_results.append(exit_type)

# Save the descriptions alongside each episode.
low_price_episodes["entry_type"] = entry_results
low_price_episodes["exit_type"] = exit_results

# ------------------------------------------
# VERIFY THAT ALL RECORDS ARE ACCOUNTED FOR
# ------------------------------------------

assert len(low_price_episodes) == 224

assert low_price_episodes["episode_weeks"].sum() == 331

assert low_price_episodes["entry_type"].notna().all()
assert low_price_episodes["exit_type"].notna().all()

print("1. EPISODE ENTRY TYPES")

display(
    low_price_episodes["entry_type"]
    .value_counts()
    .to_frame("episodes")
)

print("\n2. EPISODE EXIT TYPES")

display(
    low_price_episodes["exit_type"]
    .value_counts()
    .to_frame("episodes")
)

print("\n3. ENTRY VS EXIT")

display(
    pd.crosstab(
        low_price_episodes["entry_type"],
        low_price_episodes["exit_type"]
    )
)

print("\n4. LONGEST EPISODES")

display(
    low_price_episodes[
        [
            "item_id",
            "store_id",
            "episode_weeks",
            "entry_type",
            "exit_type"
        ]
    ]
    .sort_values("episode_weeks", ascending=False)
    .head(10)
)

print("\nAll 224 episodes and 331 low-price records accounted for.")


1. EPISODE ENTRY TYPES


,episodes
entry_type,
Large drop event,218
Other entry,5
Series start,1



2. EPISODE EXIT TYPES


,episodes
exit_type,
Other exit,217
Large increase event,7



3. ENTRY VS EXIT


exit_type,Large increase event,Other exit
entry_type,,
Large drop event,5,213
Other entry,2,3
Series start,0,1



4. LONGEST EPISODES


,item_id,store_id,episode_weeks,entry_type,exit_type
0,HOBBIES_1_234,CA_1,19,Large drop event,Large increase event
75,FOODS_3_454,TX_2,18,Other entry,Large increase event
98,FOODS_3_669,TX_2,12,Large drop event,Large increase event
192,FOODS_3_711,TX_3,9,Large drop event,Large increase event
126,FOODS_3_070,TX_3,7,Large drop event,Large increase event
221,HOBBIES_2_128,WI_2,6,Large drop event,Large increase event
133,FOODS_3_234,TX_3,4,Other entry,Large increase event
90,FOODS_3_547,TX_2,3,Large drop event,Other exit
101,FOODS_3_711,TX_2,3,Large drop event,Other exit
151,FOODS_3_454,TX_3,3,Large drop event,Other exit



All 224 episodes and 331 low-price records accounted for.


In [20]:

# ============================================================
# PRICE AUDIT — STEP 3A: REFINE EPISODE EXIT LABELS
# ============================================================

import numpy as np
import pandas as pd

# Use the original episode boundaries calculated earlier.
series_values = price_audit["series_id"].to_numpy()
price_values = price_audit["sell_price"].to_numpy()
week_values = price_audit["week_order"].to_numpy()

refined_exits = []
exit_ratios = []

for episode in low_price_episodes.itertuples():

    last_row = int(episode.last_row)
    next_row = last_row + 1

    # If there is no later price for the same product-store
    # combination, the episode ends at the dataset boundary.
    if (
        next_row >= len(price_audit)
        or series_values[next_row] != series_values[last_row]
    ):
        refined_exits.append("Series end (censored)")
        exit_ratios.append(np.nan)
        continue

    # Our earlier continuity check confirmed that
    # consecutive observed records are consecutive weeks.
    assert week_values[next_row] == week_values[last_row] + 1

    previous_low_price = price_values[last_row]
    next_price = price_values[next_row]

    ratio = next_price / previous_low_price
    exit_ratios.append(ratio)

    # Classify the observed increase.
    if (
        price_audit.iloc[next_row]["large_price_event_flag"]
        and ratio >= 2.0
    ):
        refined_exits.append("Large increase event")

    elif ratio > 1.0:
        refined_exits.append("Modest rise / other increase")

    else:
        refined_exits.append("Other exit")

# Save the refined results.
low_price_episodes["refined_exit_type"] = refined_exits
low_price_episodes["exit_price_ratio"] = exit_ratios

# Confirm no episodes were lost.
assert len(low_price_episodes) == 224
assert low_price_episodes["episode_weeks"].sum() == 331

print("1. REFINED EXIT CLASSIFICATION")

display(
    low_price_episodes["refined_exit_type"]
    .value_counts()
    .to_frame("episodes")
)

print("\n2. EXIT PRICE RATIOS")

display(
    low_price_episodes["exit_price_ratio"]
    .describe(percentiles=[0.25, 0.50, 0.75])
    .round(3)
)

print("\n3. ORIGINAL VS REFINED EXIT")

display(
    pd.crosstab(
        low_price_episodes["exit_type"],
        low_price_episodes["refined_exit_type"]
    )
)

print("\n4. FIVE OTHER-ENTRY EPISODES")

display(
    low_price_episodes.loc[
        low_price_episodes["entry_type"] == "Other entry",
        [
            "item_id",
            "store_id",
            "episode_weeks",
            "minimum_price",
            "maximum_price",
            "refined_exit_type"
        ]
    ]
)

print("\nAll 224 episodes verified.")


1. REFINED EXIT CLASSIFICATION


,episodes
refined_exit_type,
Modest rise / other increase,217
Large increase event,7



2. EXIT PRICE RATIOS


count    224.000
mean      16.824
std       76.545
min        1.625
25%        2.500
50%        2.857
75%        4.000
max      898.000
Name: exit_price_ratio, dtype: float64


3. ORIGINAL VS REFINED EXIT


refined_exit_type,Large increase event,Modest rise / other increase
exit_type,,
Large increase event,7,0
Other exit,0,217



4. FIVE OTHER-ENTRY EPISODES


,item_id,store_id,episode_weeks,minimum_price,maximum_price,refined_exit_type
72,FOODS_3_234,TX_2,2,0.07,0.07,Modest rise / other increase
75,FOODS_3_454,TX_2,18,0.10,0.10,Large increase event
80,FOODS_3_498,TX_2,1,0.07,0.07,Modest rise / other increase
114,FOODS_3_007,TX_3,1,0.07,0.07,Modest rise / other increase
133,FOODS_3_234,TX_3,4,0.10,0.10,Large increase event



All 224 episodes verified.


In [5]:

# ============================================================
# PRICE AUDIT — STEP 3B: STORE AND PRODUCT CONCENTRATION
# ============================================================

# STEP 1: Count episodes and low-price weeks in each store.
store_concentration = (
    low_price_episodes
    .groupby("store_id")
    .agg(
        episodes=("low_price_episode_id", "count"),
        low_price_weeks=("episode_weeks", "sum"),
        unique_products=("item_id", "nunique")
    )
    .sort_values("episodes", ascending=False)
)

store_concentration["episode_share_pct"] = (
    store_concentration["episodes"]
    / len(low_price_episodes) * 100
)

print("1. LOW-PRICE EPISODES BY STORE")
display(store_concentration.round(2))

# STEP 2: Identify products with repeated low-price episodes.
product_concentration = (
    low_price_episodes
    .groupby("item_id")
    .agg(
        episodes=("low_price_episode_id", "count"),
        low_price_weeks=("episode_weeks", "sum"),
        stores_affected=("store_id", "nunique")
    )
    .sort_values("episodes", ascending=False)
)

print("\n2. PRODUCTS WITH THE MOST LOW-PRICE EPISODES")
display(product_concentration.head(10))

# STEP 3: Check whether a few stores dominate.
print("\n3. CONCENTRATION SUMMARY")

print(
    "Total episodes:",
    len(low_price_episodes)
)

print(
    "Total low-price records:",
    int(low_price_episodes["episode_weeks"].sum())
)

print(
    "Stores affected:",
    len(store_concentration)
)

print(
    "Unique products affected:",
    len(product_concentration)
)

print(
    "Episodes in top 3 stores:",
    int(store_concentration["episodes"].head(3).sum())
)

print(
    "Top 3 stores' share (%):",
    round(
        store_concentration["episodes"].head(3).sum()
        / len(low_price_episodes) * 100,
        2
    )
)

# STEP 4: Verify that all episodes are represented.
assert store_concentration["episodes"].sum() == 224
assert store_concentration["low_price_weeks"].sum() == 331

print("\nAll concentration checks passed.")


NameError: name 'low_price_episodes' is not defined

In [21]:

# ============================================================
# STEP 4A — COMPARE EXITS WITH PRE-ENTRY PRICES
# ============================================================

import numpy as np
import pandas as pd

# Use the existing price histories and episode boundaries.
all_prices = price_audit["sell_price"].to_numpy()
all_series = price_audit["series_id"].to_numpy()
all_weeks = price_audit["week_order"].to_numpy()

first_rows = low_price_episodes["first_row"].to_numpy(dtype=int)
last_rows = low_price_episodes["last_row"].to_numpy(dtype=int)

# Find the week immediately BEFORE each low-price episode.
previous_rows = first_rows - 1

has_pre_entry_price = (
    (previous_rows >= 0)
    & (
        all_series[np.maximum(previous_rows, 0)]
        == all_series[first_rows]
    )
)

pre_entry_prices = np.full(len(low_price_episodes), np.nan)

pre_entry_prices[has_pre_entry_price] = (
    all_prices[previous_rows[has_pre_entry_price]]
)

# Find the week immediately AFTER each low-price episode.
next_rows = last_rows + 1

has_exit_price = (
    (next_rows < len(price_audit))
    & (
        all_series[np.minimum(next_rows, len(price_audit) - 1)]
        == all_series[last_rows]
    )
)

exit_prices = np.full(len(low_price_episodes), np.nan)

exit_prices[has_exit_price] = (
    all_prices[next_rows[has_exit_price]]
)

# Store the comparison without modifying original prices.
low_price_episodes["pre_entry_price"] = pre_entry_prices
low_price_episodes["exit_price"] = exit_prices

low_price_episodes["exit_vs_pre_entry_ratio"] = (
    low_price_episodes["exit_price"]
    / low_price_episodes["pre_entry_price"]
)

# Return means the exit price is within 5% of
# the price immediately before the low-price episode.
low_price_episodes["returned_to_pre_entry"] = (
    low_price_episodes["exit_vs_pre_entry_ratio"]
    .between(0.95, 1.05)
)

# Count how many single-week episodes returned.
single_week_return = (
    (low_price_episodes["episode_weeks"] == 1)
    & low_price_episodes["returned_to_pre_entry"]
)

print("1. RETURN TO PRE-ENTRY PRICE")

print("Total episodes:", len(low_price_episodes))

print(
    "Episodes with known pre-entry price:",
    int(has_pre_entry_price.sum())
)

print(
    "Episodes with known exit price:",
    int(has_exit_price.sum())
)

print(
    "Episodes returning within 5% of pre-entry price:",
    int(low_price_episodes["returned_to_pre_entry"].sum())
)

print(
    "Single-week episodes returning within 5%:",
    int(single_week_return.sum())
)

print("\n2. EXIT / PRE-ENTRY PRICE RATIO")

display(
    low_price_episodes["exit_vs_pre_entry_ratio"]
    .describe(percentiles=[0.25, 0.50, 0.75])
    .round(3)
)

print("\n3. LARGEST INCREASES RELATIVE TO LOW PRICE")

display(
    low_price_episodes[
        [
            "item_id",
            "store_id",
            "episode_weeks",
            "minimum_price",
            "pre_entry_price",
            "exit_price",
            "exit_price_ratio",
            "exit_vs_pre_entry_ratio",
            "returned_to_pre_entry"
        ]
    ]
    .sort_values("exit_price_ratio", ascending=False)
    .head(8)
    .round(3)
)

# Final reconciliation checks.
assert len(low_price_episodes) == 224
assert low_price_episodes["episode_weeks"].sum() == 331

# Confirm every observed episode week is still accounted for.
assert has_exit_price.sum() == 224

print("\nAll 224 episodes and 331 records remain accounted for.")


1. RETURN TO PRE-ENTRY PRICE
Total episodes: 224
Episodes with known pre-entry price: 223
Episodes with known exit price: 224
Episodes returning within 5% of pre-entry price: 219
Single-week episodes returning within 5%: 182

2. EXIT / PRE-ENTRY PRICE RATIO


count    223.000
mean       1.000
std        0.035
min        0.650
25%        1.000
50%        1.000
75%        1.000
max        1.252
Name: exit_vs_pre_entry_ratio, dtype: float64


3. LARGEST INCREASES RELATIVE TO LOW PRICE


,item_id,store_id,episode_weeks,minimum_price,pre_entry_price,exit_price,exit_price_ratio,exit_vs_pre_entry_ratio,returned_to_pre_entry
218,HOBBIES_1_338,WI_1,2,0.01,8.98,8.98,898.0,1.0,True
219,HOUSEHOLD_1_533,WI_1,1,0.01,4.97,4.97,497.0,1.0,True
222,HOUSEHOLD_1_036,WI_3,3,0.01,2.98,2.98,298.0,1.0,True
217,FOODS_3_122,WI_1,1,0.01,2.50,2.50,250.0,1.0,True
2,HOUSEHOLD_1_443,CA_1,1,0.01,2.47,2.47,247.0,1.0,True
220,FOODS_3_242,WI_2,1,0.05,9.86,9.86,197.2,1.0,True
223,HOUSEHOLD_2_182,WI_3,1,0.05,6.98,6.98,139.6,1.0,True
139,FOODS_3_410,TX_3,1,0.02,2.68,2.68,134.0,1.0,True



All 224 episodes and 331 records remain accounted for.


In [22]:

# ============================================================
# STEP 4B — TEXAS CROSS-STORE LOW-PRICE OVERLAP
# ============================================================

# STEP 1: Select low-price weekly records from Texas stores.
texas_low_prices = low_price_rows.loc[
    low_price_rows["store_id"].isin(["TX_1", "TX_2", "TX_3"])
].copy()

# STEP 2: For each product and calendar week, count
# how many different Texas stores recorded a low price.
texas_weekly_overlap = (
    texas_low_prices
    .groupby(["item_id", "wm_yr_wk"])["store_id"]
    .nunique()
    .reset_index(name="stores_with_low_price")
)

# STEP 3: Identify product-weeks with simultaneous low
# prices in at least two different Texas stores.
shared_product_weeks = texas_weekly_overlap.loc[
    texas_weekly_overlap["stores_with_low_price"] >= 2
]

# STEP 4: Mark each low-price record as shared or not shared.
texas_low_prices = texas_low_prices.merge(
    texas_weekly_overlap,
    on=["item_id", "wm_yr_wk"],
    how="left",
    validate="many_to_one"
)

texas_low_prices["shared_with_other_tx_store"] = (
    texas_low_prices["stores_with_low_price"] >= 2
)

# STEP 5: Link individual low-price records back to their
# episode IDs and determine which episodes have any overlap.
shared_episode_counts = (
    texas_low_prices
    .groupby("low_price_episode_id")["shared_with_other_tx_store"]
    .any()
)

texas_episode_ids = low_price_episodes.loc[
    low_price_episodes["store_id"].isin(["TX_1", "TX_2", "TX_3"]),
    "low_price_episode_id"
]

episodes_with_overlap = shared_episode_counts.reindex(
    texas_episode_ids,
    fill_value=False
)

print("1. TEXAS CROSS-STORE OVERLAP")

print("Texas low-price episodes:", len(texas_episode_ids))
print(
    "Episodes overlapping another Texas store:",
    int(episodes_with_overlap.sum())
)
print(
    "Episodes without cross-store overlap:",
    int((~episodes_with_overlap).sum())
)
print(
    "Share with cross-store overlap (%):",
    round(episodes_with_overlap.mean() * 100, 2)
)

print("\n2. SHARED PRODUCT-WEEKS")

print(
    "Product-week combinations shared by 2+ Texas stores:",
    len(shared_product_weeks)
)

print(
    "Product-week combinations shared by all 3 Texas stores:",
    int(
        (texas_weekly_overlap["stores_with_low_price"] == 3).sum()
    )
)

print(
    "Unique products with simultaneous low prices:",
    shared_product_weeks["item_id"].nunique()
)

# STEP 6: Verify all Texas episodes were included.
assert len(texas_episode_ids) == 211
assert len(shared_episode_counts) == 211

print("\nAll 211 Texas episodes accounted for.")


1. TEXAS CROSS-STORE OVERLAP
Texas low-price episodes: 211
Episodes overlapping another Texas store: 49
Episodes without cross-store overlap: 162
Share with cross-store overlap (%): 23.22

2. SHARED PRODUCT-WEEKS
Product-week combinations shared by 2+ Texas stores: 29
Product-week combinations shared by all 3 Texas stores: 3
Unique products with simultaneous low prices: 10

All 211 Texas episodes accounted for.


In [23]:

# ============================================================
# STEP 5 — LOW-PRICE TIERS AND RETURN EXCEPTIONS
# ============================================================

import numpy as np
import pandas as pd

# Define three price tiers.
# Round to cents to avoid floating-point comparison issues.
def classify_low_price(price):
    rounded_price = round(float(price), 2)

    if rounded_price <= 0.01:
        return "$0.01"
    elif rounded_price <= 0.05:
        return "$0.02–$0.05"
    else:
        return "$0.06–$0.10"


tier_order = ["$0.01", "$0.02–$0.05", "$0.06–$0.10"]

# PART 1: Count individual low-price weekly records by tier.
low_price_rows["price_tier"] = (
    low_price_rows["sell_price"].apply(classify_low_price)
)

weekly_tier_counts = (
    low_price_rows["price_tier"]
    .value_counts()
    .reindex(tier_order, fill_value=0)
)

# PART 2: Classify each episode using its minimum price.
low_price_episodes["minimum_price_tier"] = (
    low_price_episodes["minimum_price"].apply(classify_low_price)
)

episode_tier_summary = (
    low_price_episodes
    .groupby("minimum_price_tier")
    .agg(
        episodes=("low_price_episode_id", "count"),
        returned_within_5pct=("returned_to_pre_entry", "sum"),
        single_week_episodes=(
            "episode_weeks",
            lambda weeks: (weeks == 1).sum()
        )
    )
    .reindex(tier_order, fill_value=0)
)

episode_tier_summary["weekly_records"] = weekly_tier_counts

episode_tier_summary["episode_share_pct"] = (
    episode_tier_summary["episodes"] / 224 * 100
).round(2)

print("1. LOW-PRICE BREAKDOWN BY TIER")
display(episode_tier_summary)

# PART 3: Show episodes that have a known pre-entry
# price but did NOT return within ±5% of that price.
return_exceptions = low_price_episodes.loc[
    low_price_episodes["pre_entry_price"].notna()
    & ~low_price_episodes["returned_to_pre_entry"]
]

print("\n2. EPISODES OUTSIDE THE ±5% RETURN BAND")

display(
    return_exceptions[
        [
            "item_id",
            "store_id",
            "episode_weeks",
            "minimum_price",
            "pre_entry_price",
            "exit_price",
            "exit_vs_pre_entry_ratio"
        ]
    ].round(3)
)

# PART 4: Verify the counts match previous findings.
assert weekly_tier_counts.sum() == 331
assert episode_tier_summary["episodes"].sum() == 224
assert len(return_exceptions) == 4

print("\nAll 331 records, 224 episodes, and 4 exceptions verified.")


1. LOW-PRICE BREAKDOWN BY TIER


,episodes,returned_within_5pct,single_week_episodes,weekly_records,episode_share_pct
minimum_price_tier,,,,,
$0.01,7,7,4,12,3.12
$0.02–$0.05,76,75,60,93,33.93
$0.06–$0.10,141,137,121,226,62.95



2. EPISODES OUTSIDE THE ±5% RETURN BAND


,item_id,store_id,episode_weeks,minimum_price,pre_entry_price,exit_price,exit_vs_pre_entry_ratio
124,FOODS_3_020,TX_3,1,0.05,1.18,1.00,0.847
133,FOODS_3_234,TX_3,4,0.10,0.16,0.20,1.250
173,FOODS_3_547,TX_3,1,0.08,0.20,0.13,0.650
221,HOBBIES_2_128,WI_2,6,0.07,3.97,4.97,1.252



All 331 records, 224 episodes, and 4 exceptions verified.


In [24]:

# ============================================================
# STEP 6 — VERIFY RAW PRICE DATA INTEGRITY
# ============================================================

import pandas as pd
import hashlib

# These four columns uniquely identify a weekly price record
# and contain the original price we must preserve.
price_columns = [
    "store_id",
    "item_id",
    "wm_yr_wk",
    "sell_price"
]

# Extract the original and audited price records.
# Sorting by identifying keys makes the comparison independent
# of how the audit DataFrame was reordered.
original_prices = (
    prices[price_columns]
    .sort_values(["store_id", "item_id", "wm_yr_wk"])
    .reset_index(drop=True)
)

audited_prices = (
    price_audit[price_columns]
    .sort_values(["store_id", "item_id", "wm_yr_wk"])
    .reset_index(drop=True)
)

# Compare the entire tables, including all price values.
pd.testing.assert_frame_equal(
    original_prices,
    audited_prices,
    check_dtype=False,
    check_exact=True
)

print("1. RAW PRICE COMPARISON: PASSED")
print("Records verified:", len(original_prices))

# Calculate a reproducible SHA-256 fingerprint of the
# verified original price data in its sorted order.
# We use pandas' stable row-hashing function, then hash
# the resulting bytes with SHA-256.
row_hashes = pd.util.hash_pandas_object(
    original_prices,
    index=False
).to_numpy(dtype="uint64")

price_sha256 = hashlib.sha256(
    row_hashes.tobytes()
).hexdigest()

print("\n2. RAW PRICE FINGERPRINT")
print("SHA-256:", price_sha256)

# Final row-count check.
assert len(original_prices) == 6_841_121

print("\nAll original sell_price values remain unchanged.")


1. RAW PRICE COMPARISON: PASSED
Records verified: 6841121

2. RAW PRICE FINGERPRINT
SHA-256: 78899c748e3cbdd50868fd3b9a9e0bb92ad436a89c2a06c7d348165eea441a5e

All original sell_price values remain unchanged.


In [25]:

# ============================================================
# STEP 7 — VERIFY FORECASTING FEATURE LEAKAGE SAFETY
# ============================================================

# These columns are retrospective audit information.
# They must never be included directly as model features.
forbidden_audit_features = {
    "corrected_event_class",
    "event_class",
    "large_price_event_flag",
    "extreme_low_price_flag",
    "low_price_episode_id",
    "entry_type",
    "exit_type",
    "refined_exit_type",
    "returned_to_pre_entry",
    "pre_entry_price",
    "exit_price",
    "exit_price_ratio",
    "exit_vs_pre_entry_ratio",
    "episode_weeks",
    "minimum_price_tier",
    "shared_with_other_tx_store",
}

# These are legitimate historical price features,
# provided they are computed using only data available
# before the prediction cutoff.
potential_historical_features = {
    "previous_price",
    "reference_price_asof",
    "price_ratio_asof",
    "log_price_ratio_asof",
}

# Check whether any forecasting feature lists have
# already been defined in the notebook environment.
candidate_feature_lists = [
    "feature_columns",
    "feature_cols",
    "model_features",
    "forecast_features",
    "selected_features",
]

found_feature_lists = []

print("1. EXISTING FORECAST FEATURE LISTS")

for variable_name in candidate_feature_lists:

    if variable_name not in globals():
        continue

    feature_values = globals()[variable_name]

    # Only inspect actual collections of column names.
    if not isinstance(feature_values, (list, tuple, set)):
        continue

    if not all(isinstance(name, str) for name in feature_values):
        continue

    found_feature_lists.append(variable_name)

    forbidden_found = (
        set(feature_values) & forbidden_audit_features
    )

    print(f"\nFeature list: {variable_name}")
    print("Total features:", len(feature_values))
    print("Forbidden audit features found:", forbidden_found)

    assert not forbidden_found, (
        f"Leakage risk in {variable_name}: {forbidden_found}"
    )

# Check which retrospective columns exist in our
# audit DataFrames. Their presence is fine because
# these tables are for investigation, not model training.
print("\n2. AUDIT COLUMNS PRESENT")

for dataframe_name in [
    "price_audit",
    "events_corrected",
    "low_price_episodes",
]:

    dataframe = globals().get(dataframe_name)

    if dataframe is None:
        continue

    existing_audit_columns = (
        set(dataframe.columns) & forbidden_audit_features
    )

    print(
        f"{dataframe_name}:",
        sorted(existing_audit_columns)
    )

print("\n3. RESULT")

if found_feature_lists:
    print("Checked feature lists:", found_feature_lists)
    print("No listed forbidden audit features detected.")
else:
    print("No forecasting feature list is defined yet.")
    print("Audit labels must be excluded when features are built.")

print("\nLeakage screening complete.")


1. EXISTING FORECAST FEATURE LISTS

2. AUDIT COLUMNS PRESENT
price_audit: ['extreme_low_price_flag', 'large_price_event_flag']
events_corrected: ['corrected_event_class', 'extreme_low_price_flag', 'large_price_event_flag']
low_price_episodes: ['entry_type', 'episode_weeks', 'exit_price', 'exit_price_ratio', 'exit_type', 'exit_vs_pre_entry_ratio', 'low_price_episode_id', 'minimum_price_tier', 'pre_entry_price', 'refined_exit_type', 'returned_to_pre_entry']

3. RESULT
No forecasting feature list is defined yet.
Audit labels must be excluded when features are built.

Leakage screening complete.


## Final Price Audit — Findings and Modeling Decisions

### 1. Objective

The final price audit examined unusually low weekly selling prices and large week-to-week price changes in the M5 dataset. Its purpose was to identify potentially anomalous pricing behavior, understand whether low-price events were temporary or persistent, and establish appropriate data-preservation and forecasting-feature policies.

The investigation was descriptive. It did not attempt to establish whether unusual prices were recording errors, legitimate discounts, or clearance prices.

### 2. Data integrity and chronological validation

The dataset contains 6,841,121 weekly selling-price records across 30,490 item-store combinations.

All 6,810,631 records with a preceding price observation were separated from that observation by exactly one calendar week. No internal price-history gaps were detected.

The original price records were compared directly with the audited records after sorting by store, item, and calendar week. Every record matched exactly.

**Price-data integrity check: PASSED.**

The canonicalized price-table SHA-256 fingerprint was:

`78899c748e3cbdd50868fd3b9a9e0bb92ad436a89c2a06c7d348165eea441a5e`

This fingerprint was generated from sorted DataFrame row hashes, not the raw CSV file bytes.

### 3. Historical reference and price-event classification

Large price changes were screened using the immediately preceding observed price and a historical reference computed from earlier prices within the same item-store series.

The reference used a rolling median over up to eight preceding price observations, requiring at least four historical observations. The current price was excluded.

For retrospective event classification, the historical reference was frozen at the event onset so that subsequent price movements were evaluated against the original reference rather than a reference that adapted during the episode.

A total of **2,366 candidate price-change events** were identified and classified:

| Retrospective class | Events |
|---|---:|
| Temporary dip | 1,888 |
| Persistent change | 336 |
| Isolated spike | 29 |
| Other/unresolved | 101 |
| Incomplete follow-up | 12 |
| **Total** | **2,366** |

The persistent-change designation describes behavior over the observed follow-up window; it does not prove that the price remained at the new level indefinitely.

The 12 incomplete-follow-up events occurred within the final four observed calendar weeks and were treated separately rather than assigned an unsupported outcome.

These classifications use future observations and are therefore strictly retrospective audit labels.

### 4. Extremely low prices and consecutive episodes

An extremely low price was defined as a weekly `sell_price` at or below **$0.10**.

The audit identified:

- **331** extremely low-price weekly records.
- **224** distinct consecutive low-price episodes.
- **185** single-week episodes.
- **39** episodes lasting at least two weeks.

Episodes were defined within item-store price histories using consecutive calendar weeks. One episode could therefore contain several low-price weekly records.

Episode entries were classified as 218 large-drop events, five other-entry cases, and one episode starting at the beginning of its observed price history.

All 224 episodes had an observed exit price. None ended while still at or below $0.10 at the last recorded price. One episode had no known preceding price and was treated as left-censored for the pre-entry comparison.

### 5. Return to the pre-entry price

A **candidate temporary price anomaly** was defined as a consecutive episode with prices at or below $0.10 whose subsequent observed exit price was within ±5% of the price immediately preceding the episode.

Among 223 episodes with a known pre-entry price, **219 returned within ±5%**.

Of the 185 single-week episodes, **182 returned within ±5%**.

The median exit-price-to-pre-entry-price ratio was 1.000, with the 25th and 75th percentiles also equal to 1.000.

At these low price levels, a ±5% comparison can be extremely restrictive. When the reference price is only a few cents, the tolerance is smaller than one cent and effectively requires an exact cent-level match. The tolerance is applied to the **pre-entry price**, which may be substantially higher than the low episode price.

The four known-pre-entry episodes outside the ±5% band were:

| Item | Store | Weeks | Pre-entry price | Exit price | Exit/pre-entry |
|---|---|---:|---:|---:|---:|
| FOODS_3_020 | TX_3 | 1 | $1.18 | $1.00 | 0.847 |
| FOODS_3_234 | TX_3 | 4 | $0.16 | $0.20 | 1.250 |
| FOODS_3_547 | TX_3 | 1 | $0.20 | $0.13 | 0.650 |
| HOBBIES_2_128 | WI_2 | 6 | $3.97 | $4.97 | 1.252 |

These cases were not classified as returns to the preceding price level.

### 6. Low-price tier breakdown

Episodes were assigned a price tier using their **minimum observed price**. Weekly records were counted using **each record's own price**. Consequently, an episode can contain records from more than one tier, and the two count columns are not a one-to-one breakdown.

| Price tier | Episodes | Weekly records | Returned within ±5% |
|---|---:|---:|---:|
| $0.01 | 7 | 12 | 7 |
| $0.02–$0.05 | 76 | 93 | 75 |
| $0.06–$0.10 | 141 | 226 | 137 |
| **Total** | **224** | **331** | **219** |

Seven episodes reached $0.01, representing approximately **3.13%** of all episodes. All seven returned within ±5% of their preceding price.

One example, `HOBBIES_1_338` at `WI_1`, fell from $8.98 to $0.01 for two weeks before returning to $8.98. The resulting 898× exit-to-low-price ratio reflects a return to the previous level, not an increase to an unprecedented selling price.

Such patterns are candidates for temporary price anomalies, but no cause can be established from the price records alone.

### 7. Geographic concentration and cross-store overlap

Texas stores accounted for **211 of 224 episodes (94.2%)**, despite representing three of the dataset's ten stores and approximately 30% of item-store combinations.

The remaining 13 episodes were distributed across California and Wisconsin stores.

Cross-store overlap was defined as an episode having at least one calendar week when the same item was priced at or below $0.10 in another Texas store.

Among the 211 Texas episodes:

- **49 episodes (23.22%)** overlapped at least one other Texas store.
- **162 episodes** had no such overlap.
- **29 distinct item-calendar-week combinations** were shared by at least two Texas stores.
- **Three item-calendar-week combinations** were shared by all three Texas stores.
- **Ten distinct products** had at least one shared low-price product-week.

Product-week counts and episode counts represent different units and must not be interpreted interchangeably.

The concentration and overlap findings are descriptive. No claim is made that unusual pricing was coordinated or statistically synchronized.

### 8. Modeling and data-preservation decisions

**All original selling prices are preserved.** No low prices were automatically corrected, masked, removed, or replaced.

Historical price-derived features may be considered for forecasting only when they use information available at the relevant prediction cutoff.

Retrospective event classifications, episode outcomes, return-to-pre-entry indicators, and audit-derived labels must not be used directly as forecasting model features because they may incorporate information from later weeks.

The audit tables are retained for exploratory analysis and documentation, not as approved model-training feature tables.

No forecasting feature list has been finalized at this stage. Leakage safeguards will be enforced during later feature engineering and model training.

### 9. Final conclusion

The investigation found that most extremely low-price episodes were short interruptions followed by a return to the preceding price level. It also established substantial Texas concentration and some same-week cross-store overlap.

However, unusual pricing cannot reliably be labeled erroneous solely from the available records.

The final decision is to **preserve raw prices, document candidate anomalies, and construct future forecasting features using only information available at prediction time**.

The price audit is complete. Notebook 13 will proceed with visual exploratory data analysis and forecasting metric selection.